# Tool Calling with MCP: Travel Assistant

In `01_tool_calling.ipynb` you built a travel assistant by defining Python functions directly in the notebook.
That works well for prototypes, but the tools are **tightly coupled** to your agent code.

**Model Context Protocol (MCP)** decouples the two sides:

| | **Server** | **Client** |
|---|---|---|
| **What it is** | A separate process that owns and runs the tools | Your agent, which discovers and calls tools over the protocol |
| **Defines** | What tools exist and how they execute | Which model to use, when to call tools |
| **Knows about LLMs?** | No | Yes |
| **File** | `mcp_servers/travel_mcp_server.py` | This notebook |

Think of it like a microservice: the server **provides** an API, the client **consumes** it.
Any MCP-compatible agent, in any language or framework, can connect to the same server.

Here we connect an ADK agent to that server with `McpToolset`, which is the client: it discovers the tools and
executes them for us. Section 4 opens up the raw protocol if you want to see what it handles underneath.

⚠️ Only if you are running the code outside of Github Codespace, uncomment the following cell and run it.
This reads the `.env` file in your project and transforms its content into env variables.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()

## 1. Start the Server

`mcp_servers/travel_mcp_server.py` holds the same three travel tools from `01_tool_calling.ipynb`, now decorated
with `@mcp.tool` so FastMCP exposes them over the protocol:

```python
from fastmcp import FastMCP

mcp = FastMCP("Travel Assistant MCP Server")

@mcp.tool
def get_weather(city: str) -> dict:
    ...
```

The server has **no knowledge of LLMs or of our agent** — it just runs functions and returns results.

Start it in a terminal and leave it running for the rest of the notebook:

```bash
uv run fastmcp run exercises/01_agents/mcp_servers/travel_mcp_server.py --transport streamable-http
```

You should see the FastMCP banner and the server listening on `http://127.0.0.1:8000`.

## 2. Point an Agent at the Server

`McpToolset` is ADK's MCP client. Hand it a connection and it takes care of the rest: the handshake, asking the
server which tools exist (`tools/list`), converting their JSON schemas into declarations the model understands,
and running `tools/call` whenever the model asks for one.

- **`StreamableHTTPConnectionParams`**: *how* to reach the server. ADK also supports `StdioConnectionParams`
  (server started as a subprocess) and `SseConnectionParams`.
- **`tool_filter`**: an optional whitelist. Passing `tool_filter=["get_weather", "get_current_time"]` exposes only
  part of the server to this agent, which is how you keep a read-only agent away from write tools.

The toolset then goes straight into `tools=[...]`, in the same slot where `FunctionTool`s went in `02_travel_agent`.

In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.tools.mcp_tool import McpToolset, StreamableHTTPConnectionParams

MCP_SERVER_URL = "http://127.0.0.1:8000/mcp"

travel_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(url=MCP_SERVER_URL),
    # tool_filter=["get_weather", "get_current_time"],  # optional: expose only a subset
)

travel_agent = LlmAgent(
    name="travel_agent_via_mcp",
    model=LiteLlm("gemini/gemini-3.1-flash-lite"),
    instruction=(
        "You are a helpful travel assistant. "
        "Use the available tools to fetch current information about weather, local time, "
        "and currency conversion. Never rely on your own knowledge for these, always call a tool."
    ),
    tools=[travel_tools],
)

# Tools are discovered from the server at runtime, no schema is hardcoded here.
for tool in await travel_tools.get_tools():
    summary = (tool.description or "").split("\n")[0]
    print(f"{tool.name:20} {summary}")

## 3. Run the Agent

`Runner` drives the agent over a session and streams back events. We print every tool call and result, so you can
watch the MCP round-trips: the model decides *what* to call, `McpToolset` does the calling in the server process.

In [ ]:
import uuid

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "mcp_travel_demo"
USER_ID = "demo_user"

session_service = InMemorySessionService()
runner = Runner(agent=travel_agent, app_name=APP_NAME, session_service=session_service)


async def ask(query: str):
    """Run one query in a fresh session, printing tool calls, tool results and the final answer."""
    session_id = f"session_{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=session_id)
    message = types.Content(role="user", parts=[types.Part(text=query)])

    async for event in runner.run_async(user_id=USER_ID, session_id=session_id, new_message=message):
        if not event.content or not event.content.parts:
            continue
        for part in event.content.parts:
            if part.function_call:
                print(f"~~~ CALL    {part.function_call.name}({dict(part.function_call.args or {})})")
            elif part.function_response:
                response = dict(part.function_response.response or {})
                print(f"~~~ RESULT  {part.function_response.name} → {response.get('structuredContent', response)}")
            elif part.text and event.is_final_response():
                print(f"\n{part.text}")


await ask("""
I'm flying to Tokyo tomorrow. Can you tell me:
1. What is the current weather there?
2. What time is it in Tokyo right now?
3. How much is 500 USD in Japanese Yen?
""")

## 4. Optional: What `McpToolset` Does For You

Everything above went through ADK. To see the protocol itself, FastMCP's own `Client` speaks it directly, this is
the layer `McpToolset` wraps:

1. Client connects → MCP handshake (`async with client:`)
2. `tools/list` → the server returns the tool schemas
3. `tools/call` → the server runs the function and returns the result

Without ADK you would also convert those schemas into your LLM's tool format and write the
call-tool-then-call-model-again loop by hand, the `while` loop from `01_tool_calling.ipynb`, plus the plumbing below.

In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

client = Client(StreamableHttpTransport(url=MCP_SERVER_URL))

async with client:  # 1. handshake
    tools = await client.list_tools()  # 2. tools/list
    print("discovered :", [tool.name for tool in tools])

    result = await client.call_tool("get_weather", {"city": "Rome"})  # 3. tools/call
    print("get_weather:", result.structured_content)

### Where the pieces live

| | Tool definition | Tool execution | Agent loop |
|---|---|---|---|
| `01_tool_calling.ipynb` | In the notebook | In the notebook | Hand-rolled `litellm` loop |
| `02_travel_agent` (ADK) | In `tools.py`, as `FunctionTool` | In the agent's process | `LlmAgent` |
| **This notebook** (ADK + MCP) | In `travel_mcp_server.py`, as `@mcp.tool` | In the **server** process | `LlmAgent` |

Why the last row is the one you'd ship:

- **Tools are reusable across frameworks**: the same server could feed a LangGraph agent, a CrewAI agent, or
  Claude Desktop without a line changing in `travel_mcp_server.py`.
- **The agent stays declarative**: no bespoke tool-call loop to maintain.
- **Adding a tool is a one-side change**: restart the server and the agent picks it up, which is what the bonus
  below demonstrates.

## BONUS: Add a Travel Advisory Tool

In `01_tool_calling.ipynb` you added a `get_travel_advisory` function to the notebook itself.
Add it to the **server** this time, this is the payoff of MCP: you extend the server without touching agent code.

**Steps:**
1. Open `mcp_servers/travel_mcp_server.py` and add a new `@mcp.tool` function:

```python
@mcp.tool
def get_travel_advisory(country: str) -> dict:
    """Get the current travel advisory for a country.

    Args:
        country: The name of the country (e.g. 'Japan', 'Italy').

    Returns:
        A dict with safety_level (1-5), advisory, and entry_requirements.
    """
    advisory_data = {
        # TODO: add mock data for at least 2 countries
    }
    return advisory_data.get(
        country.lower(),
        {"safety_level": 3, "advisory": "No specific information available.", "entry_requirements": "Standard entry requirements apply."},
    )
```

2. Save the file and **restart the server** in your terminal (`Ctrl+C`, then run the `fastmcp run` command again).
3. Run the cell below. The agent rediscovers the server's tools on this run and picks up the new one —
   **the agent definition does not change.**

> The restart kills the MCP session the toolset was holding, so the cell below calls `travel_tools.close()` first.
> That drops the stale session and the toolset connects to the restarted server on the next run.
> (A `Session termination failed: 404` line in the output is that stale session being cleaned up — harmless.)
> Without it, the toolset cannot reach the old process, and the agent answers with **no tools at all**.

In [ ]:
# Drop the session pointing at the old server process, so the toolset reconnects and rediscovers
await travel_tools.close()

# The agent itself is unchanged, the new tool arrives from the server
await ask("""
I'm planning a trip to Japan next month.
What's the weather in Tokyo, what time is it there now, how much is 300 EUR in Yen,
and is Japan safe to travel to? Any entry requirements I should know about?
""")

In [ ]:
# %load ../../answers/c01_n04_mcp_and_fastmcp.py